# SUPERSTORE SALES SQL ANALYSIS

## 📌 Project Description

### This project analyzes the Kaggle Superstore dataset using SQL to uncover actionable business insights related to sales performance, customer behavior, regional profitability, and product performance. The analysis is organized into business-focused sections, including Business Overview, Customer Analysis, Product Analysis, and Advanced Business Analytics. Advanced SQL techniques such as Common Table Expressions (CTEs), Window Functions, Ranking Functions, Subqueries, and Aggregate Functions are used to solve real-world business problems and support data-driven decision-making.

## 🎯 Project Objectives
* Analyze overall sales and profitability.
* Identify top-performing products, customers, and regions.
* Evaluate customer lifetime value and purchasing behavior.
* Measure the impact of discounts on profitability.
* Detect loss-making products and improvement opportunities.
* Apply advanced SQL techniques to solve business problems.
* Generate actionable business insights and recommendations.
## 🛠️ Tools & Technologies
* SQL (SQLite)
* Jupyter Notebook
* Python (Pandas)
* SQLite3
* Kaggle Superstore Dataset(cleaned in excel)
## 📚 SQL Concepts Used
* Aggregate Functions (SUM, AVG, COUNT, MIN, MAX)
* GROUP BY & HAVING
* ORDER BY
* CASE WHEN
* Common Table Expressions (CTEs)
* Window Functions
* ROW_NUMBER()
* RANK()
DENSE_RANK()
NTILE()
PARTITION BY
COUNT(DISTINCT)
Subqueries
Profit Margin Analysis
Customer Lifetime Value (CLV)
Pareto (80/20) Analysis
📊 Business Analysis Covered
Business Overview
Customer Analysis
Product Analysis
Advanced Business Analytics
Executive KPI Analysis


## IMPORTING DATASET
DATASET IS CLEANED IN EXCEL AND IMPORTED FOR FURTHER SQL ANALYSIS

In [2]:
import pandas as pd

df = pd.read_csv("cleaned_superstore.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,Month,Month Name,Quarter
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91,2016,11,November,4
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58,2016,11,November,4
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87,2016,6,June,2
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03,2015,10,October,4
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52,2015,10,October,4


## DATABASE CONNECTION

In [3]:
import sqlite3

conn = sqlite3.connect("superstore.db")

df.to_sql(
    "cleaned_superstore",
    conn,
    if_exists="replace",
    index=False
)

9994

In [4]:
pd.read_sql(
    "SELECT * FROM cleaned_superstore LIMIT 5;",
    conn
)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,Month,Month Name,Quarter
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91,2016,11,November,4
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58,2016,11,November,4
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87,2016,6,June,2
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03,2015,10,October,4
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52,2015,10,October,4


### HELPER FUNCTION

In [5]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# BUSINESS QUESTIONS

## 📊 Part 1: Business Overview 

## Overall Business Performance
### Q1- What is the overall performance of the business in terms of sales, profit, orders, quantity sold, and average discount?

In [6]:
query = """
SELECT
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    COUNT(DISTINCT "Order ID") AS Total_Orders,
    SUM(Quantity) AS Total_Quantity_Sold,
    ROUND(AVG(Discount)*100,2) AS Avg_Discount_Percentage
FROM cleaned_superstore;
"""

run_query(query)

,Total_Sales,Total_Profit,Total_Orders,Total_Quantity_Sold,Avg_Discount_Percentage
0,2297201.07,286397.79,5009,37873,15.62


### 📈 Business Insight
* The business generated ₹22,97,201.07 in total sales across 5,009 unique orders (totaling 37,873 items sold), yielding a net profit of ₹2,86,397.79, which translates to an overall healthy profit margin of roughly 12.47%
* The average discount rate sits at 15.62% across all transactions.
  
### 💡 Recommendation
* Track these KPIs monthly to identify growth or decline early.
* While the overall profitability is positive, a 15.62% average discount is quite high. Management should audit the impact of discounting on high-volume product lines.

## Sales and Profit by Category
### Q2-Which product category contributes the most sales and profit?

In [9]:
query = """
SELECT
    Category,
    ROUND(SUM(Sales),2) AS Total_sales,
    ROUND(SUM(Profit),2) AS Total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Profit_Margin_Percentage
FROM cleaned_superstore
GROUP BY Category 
ORDER BY Total_sales DESC;
"""

run_query(query)

,Category,Total_sales,Total_profit,Profit_Margin_Percentage
0,Technology,836154.10,145455.66,17.40
1,Furniture,741999.98,18451.25,2.49
2,Office Supplies,719046.99,122490.88,17.04


### 📈 Business Insight
* Technology leads across the board, generating the highest sales (₹8,36,154.10) and the highest net profit (₹1,45,455.66) with a robust profit margin of 17.40%.
* Furniture brings in strong total sales (₹7,41,999.88, ranking second), but it suffers from extreme margin compression, yielding only ₹18,451.25 in net profit and a minimal margin of 2.49%.
* Office Supplies closely trails furniture in sales (₹7,19,046.99) but punches well above its weight in profitability, delivering ₹1,22,490.88 in net profit with a healthy margin of 17.04% (almost matching Technology).

### 💡 Recommendation
* Management must audit the pricing, discounting, and shipping costs within the Furniture category. Despite high market demand, it acts as a profit bottleneck compared to the highly efficient Technology and Office Supplies segments.

## Sales and Profit by Sub-Category
### Q3-Which sub-categories are driving profits, and which are causing losses?

In [11]:
query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Profit_Margin_Percentage
FROM cleaned_superstore
GROUP BY "Sub-Category"
ORDER BY Profit_Margin_Percentage DESC;
"""

run_query(query)

,Sub-Category,Total_Sales,Total_Profit,Profit_Margin_Percentage
0,Labels,12486.30,5546.18,44.42
1,Paper,78479.24,34053.34,43.39
2,Envelopes,16476.38,6964.10,42.27
3,Copiers,149528.01,55617.90,37.20
4,Fasteners,3024.25,949.53,31.40
5,Accessories,167380.31,41936.78,25.05
6,Art,27118.80,6527.96,24.07
7,Appliances,107532.14,18138.07,16.87
8,Binders,203412.77,30221.64,14.86
9,Furnishings,91705.12,13059.25,14.24


### 📈 Business Insight
* Copiers (₹55,617.90 profit, 37.20% margin), Accessories (₹41,936.78 profit, 25.05% margin), and Phones (₹44,516.25 profit) lead the profit generation. Meanwhile, small office essentials like Labels, Paper, and Envelopes achieve the highest percentage margins (over 42% each).
* Three sub-categories are actively bleeding money: Tables (-₹17,725.59 loss, -8.56% margin), Bookcases (-₹3,472.56 loss, -3.02% margin), and Supplies (-₹1,188.99 loss, -2.55% margin).
* This sub-category breakdown explains why the broader Furniture category struggled earlier—both Tables and Bookcases are deep in the negative, completely dragging down the profitable Furnishings sub-category.

### 💡 Recommendation
* Management must immediately review discounting rules and shipping overhead for Tables and Bookcases, as their negative margins are destroying value despite strong sales volumes. Conversely, operational investments should pivot heavily toward high-margin growth drivers like Copiers and Accessories.

## Sales and Profit by Region
### Q4-Which region contributes the highest sales and profit?

In [12]:
query = """
SELECT
    Region,
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Profit_Margin_Percentage
FROM cleaned_superstore
GROUP BY Region
ORDER BY Profit_Margin_Percentage DESC;
"""

run_query(query)

,Region,Total_Sales,Total_Profit,Profit_Margin_Percentage
0,West,725457.93,108418.79,14.94
1,East,678781.36,91522.84,13.48
2,South,391721.90,46749.71,11.93
3,Central,501239.88,39706.45,7.92


### 📈 Business Insights
* The West region leads in both total sales (₹7,25,457.93) and total profit (₹1,08,418.79), while also maintaining the highest efficiency with a profit margin of 14.94%.
* The East region closely follows with strong performance, generating ₹6,78,781.36 in sales and ₹91,522.84 in profit at a 13.48% margin.
* While the Central region ranks ahead of the South in sales volume (₹5,01,239.88 vs. ₹3,91,721.90), its profitability is heavily compressed, yielding only ₹39,706.45 in profit and a low margin of 7.92%.

### 💡 Recommendation
* Audit discounting practices, regional shipping costs, and product mix in the Central region to uncover why high sales volumes are failing to convert into healthy net profits.
* Replicate the operational and sales strategies used successfully in the West and East markets across underperforming territories to boost overall margin efficiency.

## Top 10 States by Sales
### Q5-Which states generate the highest sales revenue?

In [13]:
query = """
SELECT
    State,
    ROUND(SUM(Sales),2) AS Total_Sales
FROM cleaned_superstore
GROUP BY State
ORDER BY Total_Sales DESC
LIMIT 10;
"""

run_query(query)

,State,Total_Sales
0,California,457687.68
1,New York,310876.20
2,Texas,170187.98
3,Washington,138641.29
4,Pennsylvania,116512.02
5,Florida,89473.73
6,Illinois,80166.16
7,Ohio,78258.21
8,Michigan,76269.61
9,Virginia,70636.72


### 📈 Business Insights
* California dominates total sales by a wide margin, generating ₹4,57,687.68, followed by New York at ₹3,10,876.20. Together, these two leading states command a massive share of total enterprise revenue.
* Texas (₹1,70,187.98), Washington (₹1,38,641.29), and Pennsylvania (₹1,16,512.02) form the secondary tier of strong revenue-producing states.
* States ranging from Florida (₹89,473.73) down to Virginia (₹70,636.72) round out the top ten revenue generators, showing steady distribution across multiple geographic regions.

## 📊 PART 2: Customer Analysis 
## Top 10 Customers by Sales

### Q6-Which customers contribute the highest sales revenue?

In [14]:
query = """
SELECT
    "Customer Name",
    ROUND(SUM(Sales),2) AS Total_Sales
FROM cleaned_superstore
GROUP BY "Customer Name"
ORDER BY Total_Sales DESC
LIMIT 10;
"""

run_query(query)

,Customer Name,Total_Sales
0,Sean Miller,25043.07
1,Tamara Chand,19052.22
2,Raymond Buch,15117.35
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.34
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.08


### 📈 Business Insights
* Sean Miller stands out as the highest-contributing individual customer, generating ₹25,043.07 in total sales revenue.
* Tamara Chand follows as the second-highest contributor at ₹19,052.22, with Raymond Buch (₹15,117.35) and Tom Ashbrook (₹14,595.62) rounding out the top tier of buyers.
* The remaining top customers—ranging from Adrian Barton (₹14,473.57) down to Christopher Conant (₹12,129.08)—demonstrate a relatively steady progression of high-value individual accounts driving recurring enterprise revenue.

### 💡 Recommendation
* Enroll top clients like Sean Miller and Tamara Chand into a dedicated loyalty or VIP tier to secure long-term retention and mitigate the risk of losing high-value accounts to competitors.
* Cross-reference these top revenue-generating customers against individual profit metrics to verify that high sales volume is not being driven by excessive discounting or low-margin product purchases.

## Top 10 Customers by Profit

### Q7-Which customers generate the highest profit for the company?

In [15]:
query = """
SELECT
    "Customer Name",
    ROUND(SUM(Profit),2) AS Total_Profit
FROM cleaned_superstore
GROUP BY "Customer Name"
ORDER BY Total_Profit DESC
LIMIT 10;
"""

run_query(query)

,Customer Name,Total_Profit
0,Tamara Chand,8981.32
1,Raymond Buch,6976.09
2,Sanjit Chand,5757.42
3,Hunter Lopez,5622.43
4,Adrian Barton,5444.81
5,Tom Ashbrook,4703.80
6,Christopher Martinez,3899.91
7,Keith Dawkins,3038.58
8,Andy Reiter,2884.61
9,Daniel Raglin,2869.08


### 📈 Business Insights
* Tamara Chand leads all customers in total profit generation, delivering ₹8,981.32 to the bottom line.
* Raymond Buch follows as the second-highest profit contributor at ₹6,976.09, closely followed by Sanjit Chand (₹5,757.42) and Hunter Lopez (₹5,622.43).
* While Sean Miller generated the highest total sales revenue previously, Tamara Chand takes the #1 spot for actual net profit, proving her purchases consist of higher-margin items with less discounting.

### 💡 Recommendation
* Shift customer acquisition and account management metrics to reward sales teams based on net profit contribution rather than top-line revenue alone, using profiles like Tamara Chand as the benchmark.
* Design tailored rewards, exclusive service tiers, and proactive account management for the top ten profitable clients to ensure long-term retention and maximize lifetime value.

## Sales and Profit by Customer Segment

### Q8-How do different customer segments perform in terms of sales and profitability?

In [16]:
query = """
SELECT
    Segment,
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin
FROM cleaned_superstore
GROUP BY Segment
ORDER BY Total_Sales DESC;
"""

run_query(query)

,Segment,Total_Sales,Total_Profit,Profit_Margin
0,Consumer,1161401.34,134119.33,11.55
1,Corporate,706146.44,91979.45,13.03
2,Home Office,429653.29,60299.01,14.03


### 📈 Business Insights
* The Consumer segment leads by a wide margin in both total sales (₹11,61,401.34) and total profit (₹1,34,119.33), representing the core revenue driver for the enterprise.
* While the Corporate segment (₹7,06,146.44 sales; ₹91,979.45 profit) and Home Office segment (₹4,29,653.29 sales; ₹60,299.01 profit) have lower overall revenue footprints, they exhibit superior efficiency with higher profit margins of 13.03% and 14.03%, respectively, compared to the Consumer segment's 11.55%.

### 💡 Recommendation
* Allocate additional marketing and sales resources to scale the Corporate and Home Office segments, leveraging their stronger profit margins to enhance overall enterprise profitability.
* Conduct a thorough pricing and discount review within the high-volume Consumer segment to close the margin gap relative to corporate and home office clients.

## Average Order Value by Customer Segment

### Q9-Which customer segment places the highest-value orders on average?

In [17]:
query = """
SELECT
    Segment,
    ROUND(SUM(Sales) / COUNT(DISTINCT "Order ID"),2) AS Avg_Order_Value
FROM cleaned_superstore
GROUP BY Segment
ORDER BY Avg_Order_Value DESC;
"""

run_query(query)

,Segment,Avg_Order_Value
0,Home Office,472.67
1,Corporate,466.41
2,Consumer,449.11


### 📈Business Insights
* The Home Office segment places the highest-value orders on average, recording an Average Order Value (AOV) of ₹472.67.
* The Corporate segment follows closely behind with an average order value of ₹466.41, maintaining strong basket sizes per transaction.
* The Consumer segment ranks lowest in average order value at ₹449.11, despite previously dominating total cumulative sales and profit volume.

### 💡 Recommendation
* Design targeted product bundles and cross-selling promotions for the Consumer segment to elevate their average basket size closer to corporate levels.
* Continue incentivizing Home Office and Corporate buyers with volume discounts or loyalty perks to capitalize on their inherently larger transaction sizes.

### Most Loyal Customers (Repeat Orders)

### Q10-Which customers have placed the highest number of orders?

In [18]:
query = """
SELECT
    "Customer Name",
    COUNT(DISTINCT "Order ID") AS Total_Orders,
    ROUND(SUM(Sales),2) AS Total_Sales
FROM cleaned_superstore
GROUP BY "Customer Name"
ORDER BY Total_Orders DESC
LIMIT 10;
"""

run_query(query)

,Customer Name,Total_Orders,Total_Sales
0,Emily Phan,17,5478.06
1,Zuschuss Carroll,13,8025.70
2,Sally Hughsby,13,3406.86
3,Patrick Gardner,13,3086.90
4,Noel Staavos,13,2964.82
5,Joel Eaton,13,6760.81
6,Erin Ashbrook,13,2846.71
7,Chloris Kastensmidt,13,3154.83
8,Suzanne McNair,12,5563.40
9,Sanjit Jacobs,12,3949.65


### 📈 Business Insights
* Emily Phan places the highest number of distinct orders, recording 17 separate transactions totaling ₹5,478.06 in sales.
* A large group of customers—including Zuschuss Carroll, Sally Hughsby, Patrick Gardner, Noel Staavos, Joel Eaton, Erin Ashbrook, and Chloris Kastensmidt—tie closely behind with 13 orders each.
* Customers with high order frequencies (such as Emily Phan with 17 orders) do not necessarily match the top revenue or profit contributors analyzed previously, indicating a pattern of frequent, smaller-basket purchases.

### 💡 Recommendation
* Implement a tiered rewards or subscription-style program tailored for high-frequency buyers like Emily Phan to encourage continued repeat purchasing.
* Review shipping and fulfillment costs for frequent, lower-value order profiles to ensure that high transaction frequency translates into healthy net margins rather than eating into profits via logistics overhead.

## 📦 PART 3: Product Analysis

### Top 10 Best-Selling Products

### Q11-Which products generate the highest sales revenue?

In [19]:
query = """
SELECT
    "Product Name",
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin
FROM cleaned_superstore
GROUP BY "Product Name"
ORDER BY Total_Sales DESC
LIMIT 10;
"""

run_query(query)

,Product Name,Total_Sales,Total_Profit,Profit_Margin
0,Canon imageCLASS 2200 Advanced Copier,61599.83,25199.94,40.91
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.06,28.24
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,-8.00
3,HON 5400 Series Task Chairs for Big and Tall,21870.57,0.00,0.00
4,GBC DocuBind TL300 Electric Binding System,19823.48,2233.50,11.27
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98,4.00
6,Hewlett Packard LaserJet 3310 Copier,18839.68,6983.89,37.07
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,22.29
8,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,-10.45
9,High Speed Automatic Electric Letter Opener,17030.31,-262.00,-1.54


### 📈 Business Insights
* The Canon imageCLASS 2200 Advanced Copier dominates all top-performing products, generating a massive ₹61,599.83 in sales and ₹25,199.94 in profit, while maintaining an exceptional profit margin of 40.91%.
* Items like the Fellowes PB500 Electric Punch Plastic Comb Binding System (₹27,453.38 sales; 28.24% margin) and the Hewlett Packard LaserJet 3310 Copier (₹18,839.68 sales; 37.07% margin) deliver strong financial returns.
* Despite high sales volumes, products like the Cisco TelePresence System EX90 Videoconferencing System (-₹1,811.08 profit, -8.00% margin) and GBC DocuBind P400 Electric Binding System (-₹1,878.17 profit, -10.45% margin) are net loss-makers due to negative margins. Additionally, the HON 5400 Series Task Chairs for Big and Tall yields zero profit (0.00% margin) despite generating ₹21,870.57 in sales.

### 💡 Recommendation
* Expand inventory and marketing efforts around high-yield office machinery like the Canon imageCLASS 2200 and HP LaserJet Copiers to maximize bottom-line returns.
* Immediately review pricing, discounting structures, and vendor costs for top-selling items that generate negative margins, such as the Cisco TelePresence System and GBC DocuBind P400, or discontinue them if profitability cannot be recovered.

### Bottom 10 Products by Profit

### Q12-Which products generate the lowest profit or losses?

In [20]:
query = """
SELECT
    "Product Name",
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin
FROM cleaned_superstore
GROUP BY "Product Name"
ORDER BY Total_Profit ASC
LIMIT 10;
"""

run_query(query)

,Product Name,Total_Sales,Total_Profit,Profit_Margin
0,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97,-80.00
1,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,-27.27
2,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99,-48.00
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.11,-29.00
4,Bush Advantage Collection Racetrack Conference...,9544.72,-1934.40,-20.27
5,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,-10.45
6,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,-8.00
7,Martin Yale Chadless Opener Electric Letter Op...,16656.21,-1299.19,-7.80
8,Balt Solid Wood Round Tables,6518.76,-1201.06,-18.42
9,BoxOffice By Design Rectangular and Half-Moon ...,1706.26,-1148.44,-67.31


### 📈 Business Insights
* The Cubify CubeX 3D Printer Double Head Print is the single largest loss-maker, recording a massive enterprise loss of -₹8,879.97 at an extreme negative margin of -80.00%. Its variant, the Cubify CubeX 3D Printer Triple Head Print, also generates heavy losses (-₹3,839.99 loss, -48.00% margin).
* The Lexmark MX611dhe Monochrome Laser Printer incurs significant losses totaling -₹4,589.97 despite achieving substantial top-line sales of ₹16,829.90
* Heavy items such as the Chromcraft Bull-Nose Wood Oval Conference Table (-₹2,876.11 loss) and Bush Advantage Collection Racetrack Conference Table (-₹1,934.40 loss) consistently erode profitability due to negative margins exceeding -20%.

### 💡 Recommendation
* Immediately review and halt sales of heavily defective margin-destroyers like the Cubify CubeX 3D Printer models, as their extreme negative margins (-48% to -80%) make them impossible to sell profitably under current pricing models.
* Audit the shipping, handling, and heavy discounting associated with bulky furniture items like conference tables to mitigate ongoing logistical drains on the bottom line.

### Products Receiving the Highest Discounts
### Q13-Which products receive the highest average discount?

In [21]:
query = """
SELECT
    "Product Name",
    ROUND(AVG(Discount)*100,2) AS Avg_Discount,
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit
FROM cleaned_superstore
GROUP BY "Product Name"
ORDER BY Avg_Discount DESC
LIMIT 10;
"""

run_query(query)

,Product Name,Avg_Discount,Total_Sales,Total_Profit
0,Eureka Disposable Bags for Sanitaire Vibra Gro...,80.00,1.62,-4.47
1,GBC Plasticlear Binding Covers,72.86,68.88,-68.43
2,GBC VeloBinder Electric Binding Machine,72.50,496.02,-411.33
3,Zebra GK420t Direct Thermal/Thermal Transfer P...,70.00,703.71,-938.28
4,Plantronics Single Ear Headset,70.00,29.93,-21.95
5,Okidata B401 Printer,70.00,179.99,-251.99
6,Lexmark MarkNet N8150 Wireless Print Server,70.00,723.51,-506.46
7,Hewlett-Packard Deskjet F4180 All-in-One Color...,70.00,101.99,-71.40
8,Epson Perfection V600 Photo Scanner,70.00,206.99,-172.49
9,Cisco 8961 IP Phone Charcoal,70.00,224.94,-164.95


### 📈 Business Insights
* Every single product in the top 10 most heavily discounted list generates a net financial loss. There is a direct, destructive correlation between aggressive discounting (70%+) and bottom-line deficits.
* The top 10 products all suffer from average discounts of 70% to 80%. The Eureka Disposable Bags for Sanitaire Vibra Gro... leads with an extreme 80.00% discount, resulting in virtually zero revenue (₹1.62) and a net loss (-₹4.47).
* Higher-ticket technology items are hemorrhaging profit under these discount strategies. For example, the Zebra GK420t Direct Thermal Printer and Lexmark MarkNet N8150 generate ₹703.71 and ₹723.51 in sales respectively (at 70% average discounts), but result in disproportionately massive losses of -₹938.28 and -₹506.46.

### 💡 Recommendation
* Immediately institute a systemic cap on maximum allowable discounts (e.g., 20% to 30%) for all sales teams. Discounts of 70% and above are mathematically incompatible with healthy profit margins and must require executive approval.

* Investigate why these specific SKUs are being marked down so heavily. If this is a deliberate liquidation strategy for obsolete dead stock, ensure these products are marked as "do not restock." If these are standard promotional discounts, the promotional strategy is fundamentally flawed and requires an immediate overhaul.

### Profitability by Sub-Category

### Q14-Which sub-categories generate the highest profit margin?

In [22]:
query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit,
    ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin
FROM cleaned_superstore
GROUP BY "Sub-Category"
ORDER BY Profit_Margin DESC;
"""

run_query(query)

,Sub-Category,Total_Sales,Total_Profit,Profit_Margin
0,Labels,12486.30,5546.18,44.42
1,Paper,78479.24,34053.34,43.39
2,Envelopes,16476.38,6964.10,42.27
3,Copiers,149528.01,55617.90,37.20
4,Fasteners,3024.25,949.53,31.40
5,Accessories,167380.31,41936.78,25.05
6,Art,27118.80,6527.96,24.07
7,Appliances,107532.14,18138.07,16.87
8,Binders,203412.77,30221.64,14.86
9,Furnishings,91705.12,13059.25,14.24


### 📈 Business Insights
* The Labels sub-category achieves the highest profit margin overall at 44.42%, followed closely by Paper (43.39%) and Envelopes (42.27%).
* Copiers stand out by combining massive scale (₹1,49,528.01 in total sales and ₹55,617.90 in total profit) with a strong profit margin of 37.20%. Similarly, Accessories generate substantial revenue (₹1,67,380.31) with a healthy 25.05% margin.
* High-revenue sub-categories such as Tables (-8.56% margin, -₹17,725.59 loss), Bookcases (-3.02% margin), and Supplies (-2.55% margin) are net loss-makers, actively eroding enterprise profitability despite pulling in significant top-line figures.

### 💡 Recommendation
* Increase marketing and promotional focus on high-margin, low-risk sub-categories like Labels, Paper, and Envelopes to boost overall baseline profitability with minimal operational drag.
* Conduct an urgent cost-and-pricing audit on Tables and Bookcases to eliminate negative margins, restricting heavy discounts and addressing logistical overhead, or phase them out if profitability cannot be restored.

### Quantity Sold by Sub-Category
### Q15-Which sub-categories sell the highest number of units?

In [23]:
query = """
SELECT
    "Sub-Category",
    SUM(Quantity) AS Total_Quantity,
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit
FROM cleaned_superstore
GROUP BY "Sub-Category"
ORDER BY Total_Quantity DESC;
"""

run_query(query)

,Sub-Category,Total_Quantity,Total_Sales,Total_Profit
0,Binders,5974,203412.77,30221.64
1,Paper,5178,78479.24,34053.34
2,Furnishings,3563,91705.12,13059.25
3,Phones,3289,330007.10,44516.25
4,Storage,3158,223843.59,21279.05
5,Art,3000,27118.80,6527.96
6,Accessories,2976,167380.31,41936.78
7,Chairs,2356,328449.13,26590.15
8,Appliances,1729,107532.14,18138.07
9,Labels,1400,12486.30,5546.18


### 📈 Business Insights
* The Binders sub-category sells the highest number of units by a wide margin, moving a total quantity of 5,974 items, which translates into strong revenue (₹2,03,412.77) and healthy profit (₹30,221.64).
* Paper secures the second position in unit volume with 5,178 units sold, generating ₹78,479.24 in sales and a robust profit of ₹34,053.34. Other high-quantity drivers include Furnishings (3,563 units), Phones (3,289 units), and Storage (3,158 units).
* Capital-intensive or heavy items record the lowest unit quantities sold—such as Copiers (234 units) and Machines (440 units)—yet these low-volume items manage to generate massive revenue and substantial profit margins due to high unit price tags.

### 💡 Recommendation
* Maintain high stock availability and streamline warehouse fulfillment for top-moving unit drivers like Binders and Paper to prevent stockouts and capitalize on steady, high-frequency customer demand.
* Evaluate handling and storage efficiencies for bulky high-quantity categories like Furnishings and Storage to ensure that high unit movement scales efficiently without inflating operational overhead.

# 📌 Key Takeaways from Product Analysis

- Identified the top-performing products driving revenue.
- Highlighted loss-making products impacting profitability.
- Evaluated the relationship between discounts and profit.
- Compared sub-categories using both revenue and profit margin.
- Distinguished between high-volume and high-value product categories.

## 📅 📊 PART 4 – Advanced Business Analytics

### Top 3 Products Within Each Category

### Q16-Which are the top 3 revenue-generating products within each category?

In [12]:
query = """
WITH ProductSales AS (

SELECT
Category,
"Product Name",
ROUND(SUM(Sales),2) AS Total_Sales,
ROUND(SUM(Profit),2) AS Total_Profit,

ROW_NUMBER() OVER(
PARTITION BY Category
ORDER BY SUM(Sales) DESC
) AS Product_Rank

FROM cleaned_superstore

GROUP BY Category,"Product Name"

)

SELECT *

FROM ProductSales

WHERE Product_Rank<=3

ORDER BY Category,Product_Rank;
"""

run_query(query)

,Category,Product Name,Total_Sales,Total_Profit,Product_Rank
0,Furniture,HON 5400 Series Task Chairs for Big and Tall,21870.57,0.00,1
1,Furniture,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.97,-669.53,2
2,Furniture,Bretford Rectangular Conference Table Tops,12995.28,-327.25,3
3,Office Supplies,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.06,1
4,Office Supplies,GBC DocuBind TL300 Electric Binding System,19823.48,2233.50,2
5,Office Supplies,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98,3
6,Technology,Canon imageCLASS 2200 Advanced Copier,61599.83,25199.94,1
7,Technology,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,2
8,Technology,Hewlett Packard LaserJet 3310 Copier,18839.68,6983.89,3


### 📈 Business Insights
* The Technology category features the top-performing revenue and profit driver across all groups—the Canon imageCLASS 2200 Advanced Copier—which pulls in a massive ₹61,599.83 in sales and ₹25,199.94 in profit (Rank 1). Additionally, the Hewlett Packard LaserJet 3310 Copier yields strong returns (₹18,839.68 sales; ₹6,983.89 profit at Rank 3). However, the Cisco TelePresence System at Rank 2 creates a notable loss (-₹1,811.08) despite high top-line revenue.
* All top three products in the Office Supplies category generate positive net profits. The Fellowes PB500 Electric Punch Plastic Comb Binding System leads with ₹27,453.38 in sales and ₹7,530.06 in profit, followed by reliable contributions from GBC binding systems.
* The Furniture category shows a concerning trend: its top revenue generators—such as the HON 5400 Series Task Chairs (₹21,870.57 sales with 0.00 profit), Riverside Palais Royal Lawyers Bookcase (-₹669.53 profit), and Bretford Rectangular Conference Table Tops (-₹327.25 profit)—fail to generate positive net returns despite significant sales volume.

### 💡 Recommendation
* Prioritize marketing, inventory allocation, and customer acquisition for proven profit drivers like the Canon imageCLASS Copier and Fellowes Binding Systems.
* Immediately investigate the cost structures, heavy discounts, and fulfillment overheads of top-ranked Furniture items, as high-revenue products in this category are currently failing to yield bottom-line profits.

### Rank States by Sales & Profit

### Q17-Which states generate the highest revenue and profit?

In [13]:
query="""

SELECT

State,

ROUND(SUM(Sales),2) AS Total_Sales,

ROUND(SUM(Profit),2) AS Total_Profit,

ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin,

RANK() OVER(
ORDER BY SUM(Sales) DESC
) AS Sales_Rank,

DENSE_RANK() OVER(
ORDER BY SUM(Profit) DESC
) AS Profit_Rank

FROM cleaned_superstore

GROUP BY State

ORDER BY Sales_Rank;

"""

run_query(query)

,State,Total_Sales,Total_Profit,Profit_Margin,Sales_Rank,Profit_Rank
0,California,457687.68,76381.60,16.69,1,1
1,New York,310876.20,74038.64,23.82,2,2
2,Texas,170187.98,-25729.29,-15.12,3,49
3,Washington,138641.29,33402.70,24.09,4,3
4,Pennsylvania,116512.02,-15560.04,-13.35,5,47
5,Florida,89473.73,-3399.25,-3.80,6,41
6,Illinois,80166.16,-12607.89,-15.73,7,46
7,Ohio,78258.21,-16971.37,-21.69,8,48
8,Michigan,76269.61,24463.15,32.07,9,4
9,Virginia,70636.72,18598.00,26.33,10,5


### 📈 Business Insights
* California and New York completely dominate national performance, leading the country in both total sales and net profit. California leads at ₹457,687.68 in sales and ₹76,381.60 in profit (Rank 1 for both), while New York follows strongly with ₹310,876.20 in sales and ₹74,038.64 in profit (Rank 2 for both), maintaining healthy profit margins of 16.69% and 23.82% respectively.
* Several states generate substantial top-line revenue but suffer massive, bottom-line losses due to negative profit margins. Texas (Rank 3 in sales at ₹1,70,187.98) incurs a staggering loss of -₹25,729.29 (-15.12% margin). Similarly, Pennsylvania (-₹15,560.04 loss), Ohio (-₹16,971.37 loss), and Illinois (-₹12,607.89 loss) drain enterprise profits despite high sales volumes.
* States like Indiana (34.33% margin), Michigan (32.07% margin), and Delaware (36.35% margin) achieve exceptional profit efficiency, punching well above their weight class in net returns relative to their overall sales volume.

### 💡 Recommendation
* Immediately investigate the root causes of negative margins in high-volume states like Texas, Ohio, and Pennsylvania. Restrict excessive discounting and review regional shipping or operational overhead that turns high sales into net losses.
* Allocate targeted marketing and sales growth initiatives toward profit-efficient states like Indiana, Michigan, and Delaware to maximize return on investment with minimal margin erosion.

### Customer Lifetime Value (CLV)

### Q18-Which customers contribute the highest lifetime revenue?

In [14]:
query="""

SELECT

"Customer Name",

COUNT(DISTINCT "Order ID") AS Total_Orders,

SUM(Quantity) AS Total_Quantity,

ROUND(SUM(Sales),2) AS Lifetime_Sales,

ROUND(SUM(Profit),2) AS Lifetime_Profit,

ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin

FROM cleaned_superstore

GROUP BY "Customer Name"

ORDER BY Lifetime_Sales DESC

LIMIT 10;

"""

run_query(query)

,Customer Name,Total_Orders,Total_Quantity,Lifetime_Sales,Lifetime_Profit,Profit_Margin
0,Sean Miller,5,50,25043.07,-1980.75,-7.91
1,Tamara Chand,5,42,19052.22,8981.32,47.14
2,Raymond Buch,6,71,15117.35,6976.09,46.15
3,Tom Ashbrook,4,36,14595.62,4703.80,32.23
4,Adrian Barton,10,73,14473.57,5444.81,37.62
5,Ken Lonsdale,12,113,14175.23,806.84,5.69
6,Sanjit Chand,9,87,14142.34,5757.42,40.71
7,Hunter Lopez,6,50,12873.30,5622.43,43.68
8,Sanjit Engle,11,78,12209.44,2650.67,21.71
9,Christopher Conant,5,34,12129.08,2177.05,17.95


### 📈 Business Insights
* Top Lifetime Revenue Leader: Sean Miller generates the highest lifetime sales across all customers at ₹25,043.07 across 5 orders, but yields a heavy net loss of -₹1,980.75 at a -7.91% profit margin, representing high revenue volume driven by negative-margin purchases.
* Tamara Chand delivers the highest lifetime profit among top earners (₹8,981.32 profit on ₹19,052.22 sales at a 47.14% margin), followed closely by Raymond Buch (₹6,976.09 profit at a 46.15% margin).
* Customers like Adrian Barton (10 orders, 73 units) and Ken Lonsdale (12 orders, 113 units) demonstrate strong long-term engagement and repeat purchasing behavior, with Adrian Barton securing a solid 37.62% profit margin (₹5,444.81 profit).

### 💡 Recommendation
* Prioritize loyalty and retention incentives for highly profitable, repeat-purchase leaders like Tamara Chand, Raymond Buch, and Adrian Barton to maximize long-term enterprise value.
* Review pricing and discount structures for high-sales customers who generate net losses (such as Sean Miller) to ensure that future high-value transactions contribute positively to the bottom line.

### High Sales but Low Profit Customers

### Q19-Which customers generate above-average sales but below-average profit?

In [15]:
query="""

WITH CustomerSummary AS(

SELECT

"Customer Name",

SUM(Sales) AS Total_Sales,

SUM(Profit) AS Total_Profit

FROM cleaned_superstore

GROUP BY "Customer Name"

)

SELECT

"Customer Name",

ROUND(Total_Sales,2) AS Total_Sales,

ROUND(Total_Profit,2) AS Total_Profit,

ROUND((Total_Profit/Total_Sales)*100,2) AS Profit_Margin

FROM CustomerSummary

WHERE

Total_Sales>(
SELECT AVG(Total_Sales)
FROM CustomerSummary
)

AND

Total_Profit<(
SELECT AVG(Total_Profit)
FROM CustomerSummary
)

ORDER BY Total_Sales DESC;

"""

run_query(query)

,Customer Name,Total_Sales,Total_Profit,Profit_Margin
0,Sean Miller,25043.07,-1980.75,-7.91
1,Becky Martin,11789.64,-1659.97,-14.08
2,John Lee,9799.94,228.92,2.34
3,Grant Thornton,9351.20,-4108.66,-43.94
4,Peter Fuller,9062.86,-614.30,-6.78
...,...,...,...,...
95,Liz Thompson,2936.26,320.98,10.93
96,Becky Castell,2933.67,251.61,8.58
97,Julie Kriz,2932.49,122.66,4.18
98,Maris LaWare,2921.51,-76.18,-2.61


### 📈 Business Insights
* High-revenue customers do not guarantee profitability. For instance, Sean Miller leads the top of this list with ₹25,043.07 in total sales, yet incurs a net loss of -₹1,980.75 at a -7.91% profit margin.
* Grant Thornton demonstrates an extreme negative margin of -43.94%, resulting in a net loss of -₹4,108.66 on ₹9,351.20 in sales. Similarly, Becky Martin records a -14.08% profit margin with a loss of -₹1,659.97 on ₹11,789.64 in sales.
* Across the top rows, major buyers are heavily discounting their baskets or purchasing low-margin/loss-making product lines (such as office furniture or heavily discounted hardware), turning strong top-line revenue into bottom-line drains.

### 💡 Recommendation
* Implement strict guardrails on maximum allowable discounts for high-volume accounts to ensure that top-line revenue generation translates into positive net margins.
* Review the specific product categories and sub-categories being purchased by loss-making high-sales clients (like Sean Miller and Grant Thornton) to reprice or restrict unprofitable items from their ordering catalogs.

### Loss Making Products

### Q20-Which products are consistently generating losses?

In [16]:
query="""

SELECT

"Product Name",

ROUND(SUM(Sales),2) AS Total_Sales,

ROUND(SUM(Profit),2) AS Total_Profit,

ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin

FROM cleaned_superstore

GROUP BY "Product Name"

HAVING SUM(Profit)<0

ORDER BY Total_Profit;

"""

run_query(query)

,Product Name,Total_Sales,Total_Profit,Profit_Margin
0,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97,-80.00
1,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,-27.27
2,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99,-48.00
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.11,-29.00
4,Bush Advantage Collection Racetrack Conference...,9544.72,-1934.40,-20.27
...,...,...,...,...
296,"Brites Rubber Bands, 1 1/2 oz. Box",13.07,-0.51,-3.90
297,Rubber Band Ball,58.34,-0.31,-0.53
298,"Acco PRESSTEX Data Binder with Storage Hooks, ...",62.95,-0.16,-0.25
299,"Avery Trapezoid Extra Heavy Duty 4"" Binders",314.54,-0.01,-0.00


### 📈 Business Insights
* The Cubify CubeX 3D Printer Double Head Print incurs the worst profit margin and absolute loss on the list, wiping out -₹8,879.97 in profit on ₹11,099.96 in sales at a staggering -80.00% margin. Its triple-head variant also loses -₹3,839.99 at a -48.00% margin.
* The Lexmark MX611dhe Monochrome Laser Printer generates ₹16,829.90 in top-line revenue but yields a heavy net loss of -₹4,589.97 (-27.27% margin), indicating severe underpricing or heavy discounting tied to hardware bundles.
* Large conference infrastructure items—such as the Chromcraft Bull-Nose Wood Oval Conference Table (-₹2,876.11 loss) and Bush Advantage Collection Racetrack Conference Table (-₹1,934.40 loss)—consistently destroy bottom-line value due to high freight and fulfillment costs relative to their sale prices.

### 💡 Recommendation
* Immediately pull or reprice extreme loss-makers like the Cubify CubeX 3D Printer series and high-value printers where discounts exceed gross margins.
* Enforce a strict minimum pricing threshold or impose specialized shipping fees on bulky items like conference tables to eliminate negative margins caused by logistics overhead.

### Regional Contribution to Total Sales

### Q21-How much does each region contribute to total company sales?

In [17]:
query="""

SELECT

Region,

ROUND(SUM(Sales),2) AS Total_Sales,

ROUND(

100.0*SUM(Sales)

/

SUM(SUM(Sales)) OVER()

,2)

AS Sales_Contribution_Percentage

FROM cleaned_superstore

GROUP BY Region

ORDER BY Total_Sales DESC;

"""

run_query(query)

,Region,Total_Sales,Sales_Contribution_Percentage
0,West,725457.93,31.58
1,East,678781.36,29.55
2,Central,501239.88,21.82
3,South,391721.90,17.05


### 📈 Business Insights
* The West and East regions drive the vast majority of top-line revenue, accounting for 31.58% (₹7,25,457.93) and 29.55% (₹6,78,781.36) of total sales respectively, combining for over 61% of national turnover.
* The Central region holds a substantial share at 21.82% (₹5,01,239.88). However, previous state-level findings show this region houses severe profit-drain states like Texas, Illinois, and Ohio, indicating that high volume here does not translate to bottom-line returns.
* The South region accounts for the lowest sales contribution at 17.05% (₹3,91,721.90), representing an underpenetrated market with potential room for optimized expansion.

### 💡 Recommendation
* Maintain strong marketing and logistical support in the West and East regions to defend market share and secure steady profitability.
* Prioritize a regional audit for the Central zone to weed out deep discounting and loss-making product lines that offset its heavy 21.82% sales contribution.

### Pareto Analysis (80/20 Rule)

### Q22-Which products contribute most of the total revenue?

In [18]:
query="""

WITH ProductSales AS(

SELECT

"Product Name",

SUM(Sales) AS Sales

FROM cleaned_superstore

GROUP BY "Product Name"

),

Pareto AS(

SELECT

"Product Name",

ROUND(Sales,2) AS Sales,

ROUND(

SUM(Sales)

OVER(
ORDER BY Sales DESC
)

,2)

AS Running_Sales,

ROUND(

100.0*

SUM(Sales)

OVER(
ORDER BY Sales DESC
)

/

SUM(Sales)

OVER()

,2)

AS Running_Percentage

FROM ProductSales

)

SELECT *

FROM Pareto

ORDER BY Sales DESC;

"""

run_query(query)

,Product Name,Sales,Running_Sales,Running_Percentage
0,Canon imageCLASS 2200 Advanced Copier,61599.83,61599.83,2.68
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,89053.21,3.88
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,111691.69,4.86
3,HON 5400 Series Task Chairs for Big and Tall,21870.57,133562.26,5.81
4,GBC DocuBind TL300 Electric Binding System,19823.48,153385.74,6.68
...,...,...,...,...
1845,Avery Hi-Liter Pen Style Six-Color Fluorescent...,7.70,2297180.14,100.00
1846,Grip Seal Envelopes,7.07,2297187.21,100.00
1847,Xerox 20,6.48,2297193.69,100.00
1848,Avery 5,5.76,2297199.45,100.00


### 📈 Business Insights
* Across 1,850 total products, a small fraction of high-performing items drives a massive share of the cumulative ₹22,97,201.07 in total top-line sales.
* Leading items like the Canon imageCLASS 2200 Advanced Copier (Sales: ₹61,599.83; Running Percentage: 2.68%) and Fellowes PB500 Electric Punch rapidly build the initial cumulative baseline percentage.
* The vast majority of the 1,850 catalog items consist of low-value, high-tail items (such as Avery 5 and Eureka Disposable Bags generating ₹5.76 and ₹1.62 respectively) that cumulatively stretch the running percentage to 100.00% without meaningfully moving core revenue.

### 💡 Recommendation
* Apply the 80/20 principle to prioritize working capital, stock availability, and promotional spend strictly on the top-tier products driving the steepest slope of running sales.
* Evaluate low-velocity, micro-sales items sitting at the bottom of the catalog to minimize holding costs, catalog clutter, and logistics overhead.

### Discount Effectiveness

### Q23-How do discounts impact sales and profit?

In [19]:
query="""

SELECT

CASE

WHEN Discount=0 THEN 'No Discount'

WHEN Discount<=0.20 THEN 'Low Discount'

WHEN Discount<=0.50 THEN 'Medium Discount'

ELSE 'High Discount'

END AS Discount_Category,

COUNT(*) AS Orders,

ROUND(AVG(Sales),2) AS Avg_Sales,

ROUND(AVG(Profit),2) AS Avg_Profit,

ROUND(AVG(Discount)*100,2) AS Avg_Discount

FROM cleaned_superstore

GROUP BY Discount_Category;

"""

run_query(query)

,Discount_Category,Orders,Avg_Sales,Avg_Profit,Avg_Discount
0,High Discount,856,75.03,-89.44,71.89
1,Low Discount,3803,222.59,26.50,19.68
2,Medium Discount,537,555.94,-109.53,36.70
3,No Discount,4798,226.74,66.90,0.00


### 📈 Business Insights
* Moderate and heavy discounting severely erodes the bottom line. Medium Discount orders (averaging a 36.70% discount rate) incur the highest average loss at -₹109.53 per order, while High Discount orders (averaging a 71.89% discount rate) yield an average loss of -₹89.44 per order.
* Healthy financial returns are concentrated strictly in lower-bracket discounting. No Discount orders drive the highest average profitability at ₹66.90 per order across 4,798 orders (Avg Sales: ₹226.74), followed by Low Discount orders (averaging a 19.68% discount) which return an average profit of ₹26.50.
* Although Medium Discount orders secure the highest average sales value (₹555.94 per order), they completely backfire by creating severe net losses, proving that top-line revenue inflation via aggressive discounting is actively destroying enterprise capital.

### 💡 Recommendation
* Implement a strict corporate policy capping maximum allowed discounts below the loss-making threshold (ideally restricting promotions to the Low Discount range of ~20% or lower).
* Discontinue broad, unearned promotional markdowns in the 30% to 70%+ brackets, as they consistently turn high gross sales into heavy net losses.

### Q24-Most Profitable Product in Every Category

In [20]:
query="""

WITH ProductProfit AS(

SELECT

Category,

"Product Name",

ROUND(SUM(Profit),2) AS Total_Profit,

ROW_NUMBER() OVER(

PARTITION BY Category

ORDER BY SUM(Profit) DESC

) AS Rank_No

FROM cleaned_superstore

GROUP BY Category,"Product Name"

)

SELECT *

FROM ProductProfit

WHERE Rank_No=1;

"""

run_query(query)

,Category,Product Name,Total_Profit,Rank_No
0,Furniture,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",1927.45,1
1,Office Supplies,Fellowes PB500 Electric Punch Plastic Comb Bin...,7753.06,1
2,Technology,Canon imageCLASS 2200 Advanced Copier,25199.94,1


### 📈 Business Insights
* The Canon imageCLASS 2200 Advanced Copier stands as the ultimate profit generator across the entire enterprise, delivering a massive ₹25,199.94 in total profit (Rank 1 in its category), vastly outperforming top products in other divisions.
* The Fellowes PB500 Electric Punch Plastic Comb Binding System acts as a dependable anchor for the Office Supplies sector, securing ₹7,753.06 in net profit (Rank 1).
* The leading product in the Furniture category—the Hon Deluxe Fabric Upholstered Stacking Chairs—generates a modest ₹1,927.45 in total profit (Rank 1), highlighting that even the best-performing furniture items yield significantly lower absolute returns compared to technology and office supplies.

### 💡 Recommendation
* Focus marketing and inventory budgets on heavy-hitting profit anchors like the Canon imageCLASS Copier and Fellowes Binding Systems to maximize bottom-line returns.
* Review the pricing and fulfillment costs of top-performing furniture lines, as their relatively low net returns indicate structural margin compression across the entire category.

### Q25: Top 5 Customers in Every Region

In [21]:
query="""

WITH CustomerSales AS(

SELECT

Region,

"Customer Name",

ROUND(SUM(Sales),2) AS Total_Sales,

ROW_NUMBER() OVER(

PARTITION BY Region

ORDER BY SUM(Sales) DESC

) AS Rank_No

FROM cleaned_superstore

GROUP BY Region,"Customer Name"

)

SELECT *

FROM CustomerSales

WHERE Rank_No<=5

ORDER BY Region,Rank_No;

"""

run_query(query)

,Region,Customer Name,Total_Sales,Rank_No
0,Central,Tamara Chand,18437.14,1
1,Central,Adrian Barton,12181.60,2
2,Central,Becky Martin,10539.90,3
3,Central,Sanjit Chand,9900.19,4
4,Central,Harry Marie,6621.47,5
5,East,Tom Ashbrook,13723.50,1
6,East,Hunter Lopez,10522.55,2
7,East,Bill Shonely,10022.29,3
8,East,Greg Tran,9382.94,4
9,East,Seth Vernon,9216.56,5


### 📈 Business Insights
* Central Region Leadership: Tamara Chand anchors the top position in the Central region with ₹18,437.14 in sales (Rank 1), followed closely by Adrian Barton at ₹12,181.60 (Rank 2) and Becky Martin at ₹10,539.90 (Rank 3).
* East Region Distribution: The East region demonstrates a tight cluster of high-value buyers led by Tom Ashbrook at ₹13,723.50 (Rank 1) and Hunter Lopez at ₹10,522.55 (Rank 2), with remaining top-5 accounts scaling down smoothly toward ₹9,216.56.
* South Region Top-Line Outlier: Sean Miller stands out as the highest regional buyer across the entire breakdown, generating ₹23,669.21 in sales within the South region (Rank 1)—though, as noted in previous customer analyses, high revenue volume in this group can heavily overlap with negative net profit margins.
* West Region Consistency: Raymond Buch leads the West region with ₹14,345.28 (Rank 1), while positions 2 through 5 (Ken Lonsdale, Edward Hooks, Jane Waco, and Karen Ferguson) maintain a steady revenue band between ₹7,182.76 and ₹8,472.39.

### 💡 Recommendation
* Tailor Regional Account Management: Design differentiated VIP retention and reward programs for top-tier regional leaders (such as Tamara Chand in the Central zone and Sean Miller in the South) while simultaneously auditing their order compositions to safeguard bottom-line margins.
* Leverage Regional Purchase Patterns: Use the localized purchasing preferences of these top 5 customer cohorts per region to optimize regional inventory stocking and targeted cross-selling campaigns.

### Q26: Compare Sub-Categories with Overall Average Profit

In [22]:
query="""

WITH AvgProfit AS(

SELECT

AVG(Profit) AS Overall_Avg_Profit

FROM cleaned_superstore

)

SELECT

"Sub-Category",

ROUND(AVG(Profit),2) AS Avg_Profit,

CASE

WHEN AVG(Profit)>
(SELECT Overall_Avg_Profit FROM AvgProfit)

THEN 'Above Average'

ELSE 'Below Average'

END AS Performance

FROM cleaned_superstore

GROUP BY "Sub-Category"

ORDER BY Avg_Profit DESC;

"""

run_query(query)

,Sub-Category,Avg_Profit,Performance
0,Copiers,817.91,Above Average
1,Accessories,54.11,Above Average
2,Phones,50.07,Above Average
3,Chairs,43.10,Above Average
4,Appliances,38.92,Above Average
5,Machines,29.43,Above Average
6,Envelopes,27.42,Below Average
7,Storage,25.15,Below Average
8,Paper,24.86,Below Average
9,Binders,19.84,Below Average


### 📈 Business Insights
* Copiers completely dominate average profitability, posting a massive ₹817.91 average profit per order—far outstripping all other sub-categories and standing as the single most lucrative product line.
* Sub-categories like Accessories (₹54.11), Phones (₹50.07), Chairs (₹43.10), Appliances (₹38.92), and Machines (₹29.43) maintain solid returns well above the baseline average.
* Several major sub-categories register negative average profits, acting as severe bottom-line drains. Tables suffer the worst average loss at -₹55.57 per order, followed by Bookcases (-₹15.23) and Supplies (-₹6.26).

### 💡 Recommendation
* Prioritize inventory replenishment and promotional focus on high-performing product groups like Copiers, Accessories, and Phones.
* Immediately review pricing, shipping costs, and discount rules for Tables, Bookcases, and Supplies to eliminate negative average profit margins across these vulnerable sub-categories.

### Q27: Product Performance Matrix

In [24]:
query="""

WITH ProductPerformance AS(

SELECT

"Product Name",

SUM(Sales) AS Sales,

SUM(Profit) AS Profit

FROM cleaned_superstore

GROUP BY "Product Name"

),

AvgValues AS(

SELECT

AVG(Sales) AS AvgSales,

AVG(Profit) AS AvgProfit

FROM ProductPerformance

)

SELECT

"Product Name",

ROUND(Sales,2) AS Sales,

ROUND(Profit,2) AS Profit,

CASE

WHEN Sales>=AvgSales
AND Profit>=AvgProfit

THEN 'High Sales High Profit'

WHEN Sales>=AvgSales
AND Profit<AvgProfit

THEN 'High Sales Low Profit'

WHEN Sales<AvgSales
AND Profit>=AvgProfit

THEN 'Low Sales High Profit'

ELSE

'Low Sales Low Profit'

END AS Performance

FROM ProductPerformance

CROSS JOIN AvgValues;

"""

run_query(query)

,Product Name,Sales,Profit,Performance
0,"""While you Were Out"" Message Book, One Form pe...",25.22,10.39,Low Sales Low Profit
1,"#10 Gummed Flap White Envelopes, 100/Box",41.30,16.77,Low Sales Low Profit
2,#10 Self-Seal White Envelopes,108.68,52.12,Low Sales Low Profit
3,"#10 White Business Envelopes,4 1/8 x 9 1/2",488.91,223.12,Low Sales High Profit
4,"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",286.67,115.30,Low Sales Low Profit
...,...,...,...,...
1845,iKross Bluetooth Portable Keyboard + Cell Phon...,477.66,115.64,Low Sales Low Profit
1846,iOttie HLCRIO102 Car Mount,215.89,-11.99,Low Sales Low Profit
1847,iOttie XL Car Mount,223.89,-50.37,Low Sales Low Profit
1848,invisibleSHIELD by ZAGG Smudge-Free Screen Pro...,442.56,171.28,Low Sales High Profit


### 📈 Business Insights
* The vast majority of tracked product lines fall under the Low Sales category bracket (spanning rows 0 through 1849), demonstrating that most SKUs generate modest individual revenue totals rather than massive blockbuster turnovers.
* Despite sharing low sales volume, products exhibit starkly divergent profitability. Items like the #10 White Business Envelopes (Sales: ₹488.91, Profit: ₹223.12) and netTALK DUO VoIP Telephone Service (Sales: ₹1,112.79, Profit: ₹430.41) successfully achieve Low Sales High Profit status.
* Conversely, certain low-to-moderate sales items suffer from negative bottom-line returns, such as the iOttie HLCRIO102 Car Mount (Profit: -₹11.99) and the iOttie XL Car Mount (Profit: -₹50.37), highlighting hidden margin leakages within specific accessories.

### 💡 Recommendation
* Prioritize inventory availability for high-efficiency items categorized as Low Sales High Profit (such as specialized envelopes and telecom services) since they yield strong profit conversion relative to their turnover.
* Audit and reprice or delist negative-profit items like the iOttie Car Mount series to prevent ongoing margin erosion from low-performing inventory.

### Q28: States with Highest Profit Margin

In [26]:
query="""

SELECT

State,

ROUND(SUM(Sales),2) AS Sales,

ROUND(SUM(Profit),2) AS Profit,

ROUND(

(SUM(Profit)/SUM(Sales))*100

,2)

AS Profit_Margin

FROM cleaned_superstore

GROUP BY State

ORDER BY Profit_Margin DESC;

"""

run_query(query)

,State,Sales,Profit,Profit_Margin
0,District of Columbia,2865.02,1059.59,36.98
1,Delaware,27451.07,9977.37,36.35
2,Minnesota,29863.15,10823.22,36.24
3,Maine,1270.53,454.50,35.77
4,Indiana,53555.36,18382.97,34.33
5,Arkansas,11678.13,4008.65,34.33
6,Georgia,49095.84,16250.08,33.10
7,Montana,5589.35,1833.32,32.80
8,Rhode Island,22627.96,7285.64,32.20
9,Michigan,76269.61,24463.15,32.07


### 📈 Business Insights
* California and New York Dominate Total Profit: California leads the entire enterprise with a massive net profit of ₹76,381.60 on ₹4,57,687.68 in total sales (Profit Margin: 16.69%), closely followed by New York generating ₹74,038.64 in net profit on ₹3,10,876.20 in sales (Profit Margin: 23.82%).
* High-Margin Efficiency Leaders: While Washington secures the third-highest absolute profit at ₹33,402.70 (Margin: 24.09%), mid-volume states like Michigan, Indiana, and Minnesota achieve exceptional bottom-line efficiency with high profit margins exceeding 32% to 36%.
* Absolute Return Powerhouses: The top 5 profit-generating states (California, New York, Washington, Michigan, and Virginia) serve as the primary profit engines of the business, collectively driving a substantial share of total earnings.

### 💡 Recommendation
* Protect High-Volume Core Markets: Safeguard promotional pricing and maintain robust supply chain support in California and New York to protect these massive absolute profit pools from margin compression.
* Replicate Efficiency Models: Study the operational drivers behind high-margin states like Indiana (34.33% margin) and Minnesota (36.24% margin) to apply similar cost-to-serve and discounting discipline across lower-performing regional operations.

### Q29: Customer Segmentation using NTILE()

In [27]:
query="""

WITH CustomerSales AS(

SELECT

"Customer Name",

SUM(Sales) AS Sales

FROM cleaned_superstore

GROUP BY "Customer Name"

)

SELECT

"Customer Name",

ROUND(Sales,2) AS Sales,

NTILE(4)

OVER(

ORDER BY Sales DESC

)

AS Customer_Tier

FROM CustomerSales;

"""

run_query(query)

,Customer Name,Sales,Customer_Tier
0,Sean Miller,25043.07,1
1,Tamara Chand,19052.22,1
2,Raymond Buch,15117.35,1
3,Tom Ashbrook,14595.62,1
4,Adrian Barton,14473.57,1
...,...,...,...
788,Roy Skaria,22.33,4
789,Mitch Gastineau,16.74,4
790,Carl Jackson,16.52,4
791,Lela Donovan,5.30,4


### 📈 Business Insights
* The elite Customer_Tier 1 group—led by top accounts like Sean Miller (Sales: ₹25,043.07) and Tamara Chand (Sales: ₹19,052.22)—captures the highest volume bracket across the enterprise's 793 total unique customer accounts.
* Conversely, Customer_Tier 4 represents the lowest quartile of spenders, featuring low-turnover accounts such as Roy Skaria (₹22.33) down to Thais Sissman (₹4.84), highlighting a wide tail of minimal-engagement buyers.
* The NTILE-based segmentation effectively buckets the customer base into quartile groups, cleanly separating high-value VIP buyers from mid-tier contributors and low-spend shoppers.

### 💡 Recommendation
* Deploy dedicated account management and VIP loyalty rewards for Tier 1 customers to protect high-revenue streams while preventing churn.
* Automate marketing and digital-only self-service channels for Tier 4 accounts to cost-effectively nurture low-spend buyers without draining high-touch sales resources.

### ⭐ Q30: Executive KPI Dashboard Query

In [28]:
query="""

WITH KPI AS(

SELECT

ROUND(SUM(Sales),2) AS Total_Sales,

ROUND(SUM(Profit),2) AS Total_Profit,

COUNT(DISTINCT "Order ID") AS Total_Orders,

COUNT(DISTINCT "Customer ID") AS Total_Customers,

SUM(Quantity) AS Total_Quantity,

ROUND(AVG(Discount)*100,2) AS Avg_Discount,

ROUND((SUM(Profit)/SUM(Sales))*100,2) AS Profit_Margin

FROM cleaned_superstore

)

SELECT *

FROM KPI;

"""

run_query(query)

,Total_Sales,Total_Profit,Total_Orders,Total_Customers,Total_Quantity,Avg_Discount,Profit_Margin
0,2297201.07,286397.79,5009,793,37873,15.62,12.47


### 📈 Business Insights
* The enterprise achieves a strong macro turnover of ₹22,97,201.07 in Total_Sales across 5,009 Total_Orders and 37,873 Total_Quantity items sold.
* Total aggregate net profit stands at ₹2,86,397.79, delivering an overall Profit_Margin of 12.47% served across a distinct base of 793 Total_Customers.
* The average corporate discount rate rests at 15.62%, maintaining a baseline structure that safely avoids the severe margin erosion seen in heavy discounting tiers.

### 💡 Recommendation
* Maintain strict oversight on discount parameters to ensure the overall 12.47% profit margin remains insulated from creeping promotional creep.
* Focus cross-selling initiatives on the established 793 customer accounts to increase average order value and enhance enterprise-wide return on sales.

## Executive Summary: Superstore Dataset Strategic Analysis
### 📈 Comprehensive Business Insights
* Macro Financial Health: The enterprise generates a solid top-line revenue of ₹22,97,201.07 across 5,009 orders and 793 unique customers, yielding a net profit of ₹2,86,397.79 and an overall profit margin of 12.47%.

* Regional & Geographic Concentration: Geographic performance is heavily polarized; core profit engines like California (₹76,381.60 profit) and New York (₹74,038.64 profit) drive the business, whereas states like Texas (-₹25,729.29) and Ohio (-₹16,971.37) suffer from severe, persistent bottom-line losses.

* The Discount Destruction Trap: Moderate to heavy promotional discounting severely damages margins; orders carrying medium-to-high discounts (30% to 70%+) flip into heavy average losses (e.g., -₹109.53 per order for medium discounts), whereas profitability is safely concentrated in No Discount and Low Discount (<20%) brackets.

* Product & Category Outliers: High-value flagship products—such as the Canon imageCLASS 2200 Copier (₹25,199.94 profit) and sub-categories like Copiers (₹817.91 average profit)—vastly outperform loss-making lines like Tables (-₹55.57 avg profit) and Bookcases (-₹15.23 avg profit).

* Customer Tier Stratification: High-value accounts led by top buyers like Sean Miller (₹25,043.07 sales) and Tamara Chand drive top-line turnover, though top-tier revenue volume must be continually audited against net margins to prevent unprofitable discounting leaks.